In [1]:
import pandas as pd
import numpy as np
from data_clean_exports import data

There are 17032 rows and 16 columns in the dataset.
There are 2924 duplicated rows.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17032 entries, 0 to 17031
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   model            17032 non-null  object
 1   price            17032 non-null  object
 2   price_score      16732 non-null  object
 3   plate            17032 non-null  object
 4   Reg. year        17032 non-null  object
 5   Fuel             17032 non-null  object
 6   KM driven        17032 non-null  object
 7   Transmission     17032 non-null  object
 8   Engine capacity  16716 non-null  object
 9   Ownership        17032 non-null  object
 10  Make year        17032 non-null  object
 11  Spare key        9886 non-null   object
 12  Reg number       9886 non-null   object
 13  Insurance        4521 non-null   object
 14  Insurance type   11667 non-null  object
 15  Source           17032 non-null  obje

c:\Users\sisfi\OneDrive\Desktop\DataA\smart-buy-classifier\notebooks\data_clean_exports.py:73: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['Ownership'] = data['Ownership'].replace(maps)


After dropping duplicates, 13970 rows remain.


c:\Users\sisfi\OneDrive\Desktop\DataA\smart-buy-classifier\notebooks\data_clean_exports.py:182: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data['Reg. year'] = pd.to_datetime(data['Reg. year']).dt.year
c:\Users\sisfi\OneDrive\Desktop\DataA\smart-buy-classifier\notebooks\data_clean_exports.py:183: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data['Make year'] = pd.to_datetime(data['Make year']).dt.year


The clean data from the previous notebook has been imported to this notebook for the purpose of feature engineering.

In [2]:
data.head()

,model,price,price_score,plate,Reg. year,Fuel,KM driven,Transmission,Engine capacity,Ownership,Make year,Insurance type,Source,variant
0,JEEP COMPASS,850000.0,STEAL DEAL,DL-10,2020,Petrol,116562,Automatic,1368.0,1,2020,3rd Party,Cars24,LONGITUDE PLUS
1,MAHINDRA XUV500,451000.0,FAIR PRICE,DL-10,2016,Diesel,238933,Automatic,1997.0,1,2016,Unknown,Cars24,W6 9
2,HYUNDAI EON,84000.0,STEAL DEAL,UP-15,2013,CNG,94507,Manual,814.0,2,2012,3rd Party,Cars24,D-LITE
3,HYUNDAI I20,181000.0,STEAL DEAL,DL-10,2013,CNG,120121,Manual,1197.0,2,2013,3rd Party,Cars24,SPORTZ
4,HONDA CITY,307000.0,STEAL DEAL,DL-4C,2014,Petrol,77884,Manual,1497.0,2,2014,Unknown,Cars24,SV


# Creating Features :

In [3]:
# Creating the car age feature :
data['car_age'] = 2026 - data['Make year']
data['reg_age'] = 2026 - data['Reg. year']


In [4]:
# creating NGT_life column

def ngt_life(fuel, car_age):
    if(fuel == 'Diesel'):
        return(10-car_age)
    elif(fuel == 'Electric'):
        return(15)
    else:
        return(15-car_age)

data['ngt_life'] = data.apply(lambda row: ngt_life(row['Fuel'], row['car_age']), axis=1)
data['ngt_critical'] = data['ngt_life'].map(lambda x: True if x<=3 else False) # binary column


In Delhi, Petrol vehicles are banned from being driven on roads after they reach an age of 15 years. For Diesel, the limit is 10 years. This is possibly a strong signal that can affect the price of a vehicle. Given two identical cars, the car with the greater NGT life is preferrable to a car with lower life simply because it can be driven for longer.

Electric vehicles have no NGT life ceiling, so their ngt_life has simply been set to 15, the maximum possible value.

In [5]:
data['frequency'] = data['model'].map(data['model'].value_counts())


In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13970 entries, 0 to 17031
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   model            13970 non-null  object 
 1   price            13970 non-null  float64
 2   price_score      7099 non-null   object 
 3   plate            13970 non-null  object 
 4   Reg. year        13970 non-null  int32  
 5   Fuel             13970 non-null  object 
 6   KM driven        13970 non-null  int64  
 7   Transmission     13970 non-null  object 
 8   Engine capacity  13970 non-null  Float64
 9   Ownership        13970 non-null  int64  
 10  Make year        13970 non-null  int32  
 11  Insurance type   13970 non-null  object 
 12  Source           13970 non-null  object 
 13  variant          13970 non-null  object 
 14  car_age          13970 non-null  int32  
 15  reg_age          13970 non-null  int32  
 16  ngt_life         13970 non-null  int64  
 17  ngt_critical     

In [7]:
#RTO feature
data['re_register'] = ~(data['plate'].astype('str').str.contains('DL|Delhi|BH'))

This feature is created based on the hypothesis that a car registered outside Delhi may cost less than a car registered within Delhi, because a driver will have to re-register the car in Delhi before he can drive it within Delhi. 

In [8]:
# km_per_year column:
data['km_per_year'] = (data['KM driven']/data['car_age']).round(2)

Gives a measure of how much a car has actually been driven over the years. It is more explainable than two individual KM and Year columns.

In [9]:
# Luxury car brands :
luxury_cars = 'VOLVO|MERCEDES|AUDI|LEXUS|ROLLS-ROYCE|BENTLEY|PORSCHE|FERRARI|LAMBORGH|ASTON MARTIN|LAND ROVER|RANGE ROVER|BMW 7|BMW X7'
data['is_luxury_brand'] = data['model'].str.contains(luxury_cars)
data['is_luxury_brand'].value_counts()

is_luxury_brand
False    13726
True       244
Name: count, dtype: int64

Luxury car brands will have a higher degree of quality and price compared to other used cars.

In [10]:
data['ownership_strength'] = data['Ownership']/data['car_age']

In [ ]:
data.to_csv('data.csv')

OSError: Cannot save file into a non-existent directory: 'streamlit'